In [ ]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from torchvision import transforms


seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

In [ ]:
DATA_ROOT = "CelebFaces"
IMAGE_SIZE = 64 
BATCH_SIZE = 64
NUM_WORKERS = 4
MAX_SAMPLES = 2000


class FaceImageFolder(Dataset):
    def __init__(self, root, transform=None, max_samples=None):
        self.paths = []
        for dirpath, dirnames, filenames in os.walk(root):
            for f in filenames:
                if f.lower().endswith((".jpg", ".jpeg", ".png")):
                    self.paths.append(os.path.join(dirpath, f))
        self.paths.sort()
        if len(self.paths) == 0:
            raise RuntimeError(f"No images found in {root}")
        if max_samples is not None:
            self.paths = self.paths[:max_samples]
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img

transform = T.Compose(
    [
        T.Resize(IMAGE_SIZE),
        T.CenterCrop(IMAGE_SIZE),
        T.ToTensor(),
        T.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
    ]
)

dataset = FaceImageFolder(DATA_ROOT, transform=transform, max_samples=MAX_SAMPLES)

loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
    drop_last=True,
)

print(len(dataset))

In [ ]:
NZ = 100  

z_demo = torch.randn(64, NZ, 1, 1)

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))

axes[0].hist(z_demo.numpy().ravel(), bins=50, density=True, color="steelblue", alpha=0.85)
xs = np.linspace(-4, 4, 200)
axes[0].plot(xs, (1 / np.sqrt(2 * np.pi)) * np.exp(-0.5 * xs**2), "r-", lw=2, label="N(0,1)")
axes[0].set_title("Гистограмма z (весь батч)")
axes[0].set_xlabel("значение")
axes[0].legend()

# Тепловая карта одного вектора z (развёртка 10x10)
v = z_demo[0].view(NZ).numpy()
axes[1].imshow(v.reshape(10, 10), cmap="coolwarm", vmin=-3, vmax=3)
axes[1].set_title("Один z как карта 10×10")
axes[1].axis("off")

axes[2].scatter(z_demo[:500, 0, 0, 0], z_demo[:500, 1, 0, 0], alpha=0.35, s=8)
axes[2].set_title("Первые 2 измерения z (500 точек)")
axes[2].set_aspect("equal", adjustable="box")
axes[2].axhline(0, color="gray", lw=0.5)
axes[2].axvline(0, color="gray", lw=0.5)
plt.tight_layout()
plt.show()

print("z_demo shape:", tuple(z_demo.shape), "| mean ≈", float(z_demo.mean()), "| std ≈", float(z_demo.std()))

In [ ]:
NC = 3   
NGF = 64  # базовая ширина генератора
NDF = 64  # базовая ширина дискриминатора


def weights_init(m):
    if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d, nn.Linear)):
        nn.init.normal_(m.weight.data, 0.0, 0.02)
        if m.bias is not None:
            nn.init.constant_(m.bias.data, 0)
    elif isinstance(m, nn.BatchNorm2d):
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)


class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.ConvTranspose2d(NZ, NGF * 8, 4, 1, 0, bias=False),
            nn.BatchNorm2d(NGF * 8),
            nn.ReLU(True),
            nn.ConvTranspose2d(NGF * 8, NGF * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(NGF * 4),
            nn.ReLU(True),
            nn.ConvTranspose2d(NGF * 4, NGF * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(NGF * 2),
            nn.ReLU(True),
            nn.ConvTranspose2d(NGF * 2, NGF, 4, 2, 1, bias=False),
            nn.BatchNorm2d(NGF),
            nn.ReLU(True),
            nn.ConvTranspose2d(NGF, NC, 4, 2, 1, bias=False),
            nn.Tanh(),
        )

    def forward(self, z):
        return self.net(z)


class Discriminator(nn.Module):

    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(NC, NDF, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(NDF, NDF * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(NDF * 2),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(NDF * 2, NDF * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(NDF * 4),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(NDF * 4, NDF * 8, 4, 2, 1, bias=False),
            nn.BatchNorm2d(NDF * 8),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(NDF * 8, 1, 4, 1, 0, bias=False),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return self.net(x)


G = Generator().to(device)
D = Discriminator().to(device)
G.apply(weights_init)
D.apply(weights_init)



In [ ]:
LR = 0.0002
BETA1 = 0.5

criterion = nn.BCELoss()
optG = optim.Adam(G.parameters(), lr=LR, betas=(BETA1, 0.999))
optD = optim.Adam(D.parameters(), lr=LR, betas=(BETA1, 0.999))


fixed_z = torch.randn(64, NZ, 1, 1, device=device)

real_label = 1.0
fake_label = 0.0

In [ ]:
EPOCHS = 5 

G_losses, D_losses = [], []


def train_epoch():
    G.train()
    D.train()
    g_run, d_run, n = 0.0, 0.0, 0
    for real in loader:
        real = real.to(device)
        bsz = real.size(0)

        D.zero_grad()
        out_real = D(real).view(-1)
        y_real = torch.full((bsz,), real_label, device=device)
        loss_d_real = criterion(out_real, y_real)
        loss_d_real.backward()

        z = torch.randn(bsz, NZ, 1, 1, device=device)
        fake = G(z).detach()
        out_fake = D(fake).view(-1)
        y_fake = torch.full((bsz,), fake_label, device=device)
        loss_d_fake = criterion(out_fake, y_fake)
        loss_d_fake.backward()
        optD.step()
        loss_d = (loss_d_real + loss_d_fake).item()

        G.zero_grad()
        z = torch.randn(bsz, NZ, 1, 1, device=device)
        fake = G(z)
        out = D(fake).view(-1)
        y = torch.full((bsz,), real_label, device=device)
        loss_g = criterion(out, y)
        loss_g.backward()
        optG.step()
        loss_g = loss_g.item()

        g_run += loss_g
        d_run += loss_d
        n += 1
    return g_run / n, d_run / n


for epoch in range(1, EPOCHS + 1):
    g_loss, d_loss = train_epoch()
    G_losses.append(g_loss)
    D_losses.append(d_loss)
    G.eval()
    with torch.no_grad():
        fakes = G(fixed_z).cpu()
    print(f"epoch {epoch}/{EPOCHS}  G_loss={g_loss:.4f}  D_loss={d_loss:.4f}")


In [ ]:
import matplotlib.pyplot as plt

EPOCHS = 5 
G_losses, D_losses = [], []

def train_epoch():
    G.train()
    D.train()
    g_run, d_run, n = 0.0, 0.0, 0
    for real in loader:
        real = real.to(device)
        bsz = real.size(0)

        D.zero_grad()
        out_real = D(real).view(-1)
        y_real = torch.full((bsz,), real_label, device=device)
        loss_d_real = criterion(out_real, y_real)
        loss_d_real.backward()

        z = torch.randn(bsz, NZ, 1, 1, device=device)
        fake = G(z).detach()
        out_fake = D(fake).view(-1)
        y_fake = torch.full((bsz,), fake_label, device=device)
        loss_d_fake = criterion(out_fake, y_fake)
        loss_d_fake.backward()
        optD.step()
        loss_d = (loss_d_real + loss_d_fake).item()

        G.zero_grad()
        z = torch.randn(bsz, NZ, 1, 1, device=device)
        fake = G(z)
        out = D(fake).view(-1)
        y = torch.full((bsz,), real_label, device=device)
        loss_g = criterion(out, y)
        loss_g.backward()
        optG.step()
        loss_g = loss_g.item()

        g_run += loss_g
        d_run += loss_d
        n += 1
    return g_run / n, d_run / n

for epoch in range(1, EPOCHS + 1):
    g_loss, d_loss = train_epoch()
    G_losses.append(g_loss)
    D_losses.append(d_loss)
    
    G.eval()
    with torch.no_grad():
        fakes = G(fixed_z).cpu()
    
    print(f"Epoch {epoch}/{EPOCHS} | G_loss: {g_loss:.4f} | D_loss: {d_loss:.4f}")
    
    fig, axes = plt.subplots(4, 4, figsize=(8, 8))
    for i, ax in enumerate(axes.flat):
        if i < len(fakes):
            img = fakes[i].permute(1, 2, 0).detach().numpy()
            img = (img + 1) / 2
            img = np.clip(img, 0, 1)
            ax.imshow(img)
        ax.axis("off")
    
    plt.suptitle(f"GAN Generated Samples - Epoch {epoch}", fontsize=12)
    plt.tight_layout()
    plt.show()
    plt.close()

print("\nTraining finished!")
print(f"Final G_loss: {G_losses[-1]:.4f}")
print(f"Final D_loss: {D_losses[-1]:.4f}")

plt.figure(figsize=(10, 5))
plt.plot(G_losses, label="Generator Loss", color="blue")
plt.plot(D_losses, label="Discriminator Loss", color="red")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("GAN Training Losses")
plt.legend()
plt.grid(True)
plt.show()
plt.close()